In [2]:
#q1
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X, y = load_wine(return_X_y=True, as_frame=True)

print("Shape of X:", X.shape)
print("Classes:")
print(y.value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Shape of X: (178, 13)
Classes:
target
0    59
1    71
2    48
Name: count, dtype: int64
Training shape: (124, 13)
Testing shape: (54, 13)
Scaled training shape: (124, 13)
Scaled testing shape: (54, 13)
Test accuracy: 0.9814814814814815
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54



In [3]:
#q2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
print("Decision Tree accuracy:", accuracy_score(y_test, tree_pred))

forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)
print("Random Forest accuracy:", accuracy_score(y_test, forest_pred))

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": forest_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("Top 3 features:")
print(importance.head(3).to_string(index=False))

Decision Tree accuracy: 0.9629629629629629
Random Forest accuracy: 1.0
Top 3 features:
        Feature  Importance
color_intensity    0.180198
     flavanoids    0.165947
        alcohol    0.142041


In [5]:
#q3
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
pred_unscaled = knn_unscaled.predict(X_test)
accuracy_unscaled = accuracy_score(y_test, pred_unscaled)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
pred_scaled = knn_scaled.predict(X_test_scaled)
accuracy_scaled = accuracy_score(y_test, pred_scaled)

print("Unscaled accuracy:", accuracy_unscaled)
print("Scaled accuracy:", accuracy_scaled)
print("Both accuracies:", accuracy_unscaled, accuracy_scaled)

Unscaled accuracy: 0.7407407407407407
Scaled accuracy: 0.9629629629629629
Both accuracies: 0.7407407407407407 0.9629629629629629


In [4]:
#q4
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd

X_all_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_all_scaled)

print("Rows in each cluster:")
print(pd.Series(labels).value_counts().sort_index())

clustered_data = X.copy()
clustered_data["cluster"] = labels

print("Cluster means:")
print(clustered_data.groupby("cluster")[
    ["alcohol", "proline", "color_intensity"]
].mean())

Rows in each cluster:
0    65
1    51
2    62
Name: count, dtype: int64
Cluster means:
           alcohol      proline  color_intensity
cluster                                         
0        12.250923   510.169231         2.973077
1        13.134118   619.058824         7.234706
2        13.676774  1100.225806         5.453548


In [1]:
#q5
import csv
import json

csv_text = """name,marks,city
Asha,78,Bangalore
Ravi,,Chennai
Meena,91,Bangalore
Imran,64,Hyderabad
Divya,88,
"""

with open("students.csv", "w", newline="") as file:
    file.write(csv_text)

def read_marks(path):
    try:
        with open(path, "r", newline="") as file:
            reader = csv.DictReader(file)
            total_rows = 0
            marks = []

            for row in reader:
                total_rows += 1
                value = row["marks"].strip()
                if value:
                    marks.append(float(value))

            count = len(marks)
            average = sum(marks) / count if count else 0

            summary = {
                "total_rows": total_rows,
                "rows_with_marks": count,
                "average_marks": average
            }

            with open("summary.json", "w") as json_file:
                json.dump(summary, json_file, indent=4)

            print("Summary:", summary)
            with open("summary.json", "r") as json_file:
                print(json_file.read())

            return summary

    except FileNotFoundError:
        print(f"File not found: {path}")
        return None

print("First call:")
read_marks("students.csv")

print("Second call:")
read_marks("missing.csv")

First call:
Summary: {'total_rows': 5, 'rows_with_marks': 4, 'average_marks': 80.25}
{
    "total_rows": 5,
    "rows_with_marks": 4,
    "average_marks": 80.25
}
Second call:
File not found: missing.csv
